In [ ]:
%load_ext autoreload
%autoreload 2

import os

import polars as pl
from sklearn.model_selection import train_test_split

from deephit_cancer_comparison.constants import DATA_PATH, GRAPH_PATH, SEED

SEER_ORIGINAL_DATA_PATH = DATA_PATH / "seer_original" / "seer_dataset.txt"
CANCER_SPECIFIC_DATA_PATH = DATA_PATH / "cancer_specific_data"

In [ ]:
SEER_MISSING_PATTERNS = [
    "Unknown",
    "NA",
    "Blank(s)",
    "unknown",
    "blank(s)",
    "blank",
    "not applicable",
    "unknown/not applicable",
    "na",
    "999",
    "9999",
    "99999",
]
seer_df = pl.read_csv(
    SEER_ORIGINAL_DATA_PATH,
    null_values=SEER_MISSING_PATTERNS,
).drop(
    pl.col("Site recode ICD-O-3 2023 Revision")  # Using the expanded version of this variable
)

In [ ]:
col_map = {
    "Race and origin recode (NHW, NHB, NHAIAN, NHAPI, Hispanic)": "race_origin",
    "Age recode with single ages and 90+": "age",
    "Combined Summary Stage with Expanded Regional Codes (2004+)": "summary_stage",
    "Histology recode - broad groupings": "histology",
    "Marital status at diagnosis": "marital_status",
    "Sequence number": "sequence_number",
    "Site recode ICD-O-3 2023 Revision Expanded": "site_recode",
    "Tumor Size Over Time Recode (1988+)": "tumor_size",
    "SEER cause-specific death classification": "cause_specific_death",
    "SEER other cause of death classification": "other_cause_death",
    "Survival months": "survival_months",
    "Survival months flag": "survival_months_flag",
    "Vital status recode (study cutoff used)": "vital_status",
    "Grade Recode (thru 2017)": "grade",
    "Year of diagnosis": "year_dx",
    "Sex": "sex",
}
seer_df = seer_df.rename(col_map)

In [ ]:
top10cancers = (seer_df.group_by("site_recode").len().sort("len", descending=True).head(10))[
    "site_recode"
].to_list()

print("=== TOP 10 MOST PREVALENT CANCERS ===")
print("\n".join(f"{cancer}" for cancer in top10cancers))

seer_df = seer_df.filter(
    pl.col("site_recode").is_in(top10cancers)
)  # Considering only top 10 most prevalent cancers

In [ ]:
print("=== SHAPE ===")
print(f"Rows: {seer_df.shape[0]:,} | Columns: {seer_df.shape[1]:,}")

In [ ]:
print("=== DTYPES ===")
print("\n".join(f"{col} | {dtype}" for col, dtype in zip(seer_df.columns, seer_df.dtypes)))

In [ ]:
n_dupes = seer_df.is_duplicated().sum()
print("=== DUPLICATES ===")
print(f"Exact duplicate rows: {n_dupes:,}")

seer_df = seer_df.unique()  # Dropping exact duplicates
print(f"After dropping duplicates: {seer_df.shape[0]:,}")

In [ ]:
print("=== MISSINGNESS AUDIT ===")
miss_rows = []
for col in seer_df.columns:
    null_count = seer_df[col].null_count()
    pct = round(100 * null_count / seer_df.shape[0], 2)
    miss_rows.append(
        {
            "column": col,
            "null_count": null_count,
            "pct_missing": pct,
        }
    )

miss_df = pl.DataFrame(miss_rows).sort("pct_missing", descending=True)
miss_df

In [ ]:
print("=== SURVIVAL MONTHS SANITY CHECK ===")
temp_df = seer_df.with_columns(pl.col("survival_months").cast(pl.Float64, strict=False))
print(f"Negative values:    {(temp_df["survival_months"] < 0).sum()}")
print(f"Zero values:        {(temp_df["survival_months"] == 0).sum()}")
print(f"Values > 300 mo:    {(temp_df["survival_months"] > 300).sum()}")
with pl.Config(set_fmt_float="full"):
    print(temp_df["survival_months"].describe())

In [ ]:
cat_cols = [
    "sex",
    "race_origin",
    "summary_stage",
    "histology",
    "marital_status",
    "sequence_number",
    "grade",
    "cause_specific_death",
    "other_cause_death",
    "vital_status",
    "survival_months_flag",
]

print("=== CATEGORICAL VALUE COUNTS ===")
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(seer_df.group_by(col).agg(pl.len().alias("count")).sort("count", descending=True))

In [ ]:
print("=== YEAR OF DIAGNOSIS DISTRIBUTION ===")
(seer_df.group_by("year_dx").agg(pl.len().alias("count")).sort("year_dx", descending=True))

In [ ]:
print("=== TUMOR SIZE DISTRIBUTION ===")
temp_df = seer_df.with_columns(pl.col("tumor_size").cast(pl.Float64, strict=False))
with pl.Config(set_fmt_float="full"):
    print(temp_df["tumor_size"].describe())

In [ ]:
print("=== CANCER TYPE COUNTS ===")
(seer_df.group_by("site_recode").agg(pl.len().alias("count")).sort("count", descending=True))

In [ ]:
def count_seer_missing(series: pl.Series) -> int:
    return (
        series.cast(pl.Utf8).str.strip_chars().str.to_lowercase().is_in(SEER_MISSING_PATTERNS).sum()
    )


print("=== MISSINGNESS PER COHORT ===")

for col in seer_df.columns:
    print(f"\n--- {col} ---")
    rows = []
    for cancer in top10cancers:
        sub = seer_df.filter(pl.col("site_recode") == cancer)
        n = sub.shape[0]
        total_miss = sub[col].null_count() + count_seer_missing(sub[col])
        pct = round(100 * total_miss / n, 2)
        rows.append({"cancer": cancer, "n": n, "missing": total_miss, "pct_missing": pct})
    print(pl.DataFrame(rows))

In [ ]:
seer_clean_df = (
    seer_df.drop(pl.col("grade"))  # Dropped due to really high missingness ratio
    .filter(
        (pl.col("survival_months").is_not_null())  # Drop rows where survival months are not known
        & (pl.col("sequence_number") == "One primary only")  # Only patients with first cancers
        & (pl.col("year_dx").ge(2004))  # Drop all patients diagnosed before 2004
    )
    .with_columns(
        pl.col("summary_stage").fill_null("Unknown"),  # Encode nulls as "Unknown"
        pl.col("marital_status").fill_null("Unknown"),  # Encode nulls as "Unknown"
    )
    .drop("sequence_number")  # Drop after filtering on
)

In [ ]:
print("=== RATIO OF DEATHS IN FIRST MONTH ===")
(
    seer_clean_df.group_by("site_recode")
    .agg(
        (pl.col("survival_months") == 0).sum().alias("zero_survival_months"),
        (pl.col("survival_months") != 0).sum().alias("not_zero_survival_months"),
    )
    .with_columns(
        (pl.col("zero_survival_months") / pl.col("not_zero_survival_months") * 100).alias(
            "pct_zero"
        )
    )
    .sort("zero_survival_months", descending=True)
)

In [ ]:
print("=== FINAL DATASET SUMMARY ===")
print(f"Shape: {seer_clean_df.shape[0]:,} rows x {seer_clean_df.shape[1]} columns")
print(f"\nColumns: {seer_clean_df.columns}")

print("\nRemaining nulls per column:")
for col in seer_clean_df.columns:
    n = seer_clean_df[col].null_count()

    print(f"  {col}: {n:,}")

print("\nCohort sizes:")
print(seer_clean_df.group_by("site_recode").agg(pl.len().alias("n")).sort("n", descending=True))

In [ ]:
cohort_df = (
    seer_clean_df.with_columns(
        pl.when(
            (pl.col("cause_specific_death") == "Dead (attributable to this cancer dx)")
            & (pl.col("vital_status") == "Dead")
        )
        .then(0)
        .when(
            (pl.col("cause_specific_death") != "Dead (attributable to this cancer dx)")
            & (pl.col("vital_status") == "Dead")
        )
        .then(1)
        .otherwise(2)
        .alias("outcome")
    )
    .group_by("site_recode")
    .agg(
        [
            pl.len().alias("cohort_size"),
            (pl.col("outcome") == 0).sum().alias("n_cancer_deaths"),
            (pl.col("outcome") == 1).sum().alias("n_other_deaths"),
            (pl.col("outcome") == 2).sum().alias("n_censored"),
        ]
    )
    .sort("cohort_size", descending=True)
)
cohort_df

In [ ]:
print(SEER_ORIGINAL_DATA_PATH)

In [ ]:
WORKING_DATA_FILES = DATA_PATH / "seer_working_data"

if not os.path.exists(WORKING_DATA_FILES):
    os.makedirs(WORKING_DATA_FILES)

seer_clean_df.write_csv(WORKING_DATA_FILES / "seer_clean.csv")

cohort_df.write_csv(WORKING_DATA_FILES / "seer_cohort.csv")

In [ ]:
# ======================================================================== #
#                   COVARIATE PREPARATION AND IMPUTATION                   #
# ======================================================================== #
YEAR_MIN = seer_clean_df["year_dx"].min()
YEAR_MAX = seer_clean_df["year_dx"].max()

seer_encoded = seer_clean_df.with_columns(
    # ==============================
    # Binary coding of SEX variabl
    # ==============================
    pl.col("sex").replace({"Male": 0, "Female": 1}).cast(pl.Int8).alias("sex"),
    # ===========================================
    # Min-max normalization for YEAR_DX -> [0, 1]
    # ===========================================
    ((pl.col("year_dx") - YEAR_MIN) / (YEAR_MAX - YEAR_MIN)).alias("year_dx"),
)

# ===============================================
# One-Hot encoding for RACE_ORIGIN (DeepHit 2018)
# ===============================================
RACE_COL = "race_origin"

RACE_CATEGORIES = seer_clean_df[RACE_COL].unique().to_list()
RACE_CATEGORIES.remove("Non-Hispanic White")

RACE_DUMMIES_MAPPING = {
    f"{RACE_COL}_Hispanic (All Races)": "race_hispanic",
    f"{RACE_COL}_Non-Hispanic American Indian/Alaska Native": "race_american_indian_alaska_native",
    f"{RACE_COL}_Non-Hispanic Asian or Pacific Islander": "race_asian_pacific_islander",
    f"{RACE_COL}_Non-Hispanic Black": "race_black",
    f"{RACE_COL}_Non-Hispanic Unknown Race": "race_unknown",
}

race_dummies = (seer_clean_df.select(pl.col(RACE_COL)).to_dummies(columns=[RACE_COL])).drop(
    f"{RACE_COL}_Non-Hispanic White"
)

race_dummies = race_dummies.with_columns([pl.col(c).cast(pl.Int8) for c in race_dummies.columns])

seer_encoded = pl.concat([seer_encoded.drop(RACE_COL), race_dummies], how="horizontal").rename(
    RACE_DUMMIES_MAPPING
)

# ========================================
# Min-max normalization for AGE -> [0, 1]
# ===========================================


def parse_age(age_str: str) -> int:
    age_str = age_str.strip().replace(" years", "").replace("+", "")
    return int(age_str)


AGE_MIN = 0
AGE_MAX = 90

seer_encoded = seer_encoded.with_columns(
    pl.col("age").map_elements(parse_age, return_dtype=pl.Int16).alias("age")
).with_columns(((pl.col("age") - AGE_MIN) / (AGE_MAX - AGE_MIN)).cast(pl.Float32).alias("age"))

# ===============================================================================================================
# Ordinal encoding for STAGE, impute unknown with mode and flag it with STAGE_UNKNOWN variable (Hernandez-Perez)
# ===============================================================================================================
STAGE_ORDER = {
    "In situ": 0,
    "Localized only": 1,
    "Regional by direct extension only": 2,
    "Regional lymph nodes involved only": 3,
    "Regional by both direct extension and lymph node involvement": 4,
    "Distant site(s)/node(s) involved": 5,
}

UNKNOWN_LABEL = "Unknown/unstaged/unspecified/DCO"
STAGE_COL = "summary_stage"

seer_encoded = seer_encoded.with_columns(
    (pl.col(STAGE_COL) == UNKNOWN_LABEL).cast(pl.Int8).alias("stage_unknown")
)

seer_encoded = seer_encoded.with_columns(
    pl.col(STAGE_COL)
    .replace(UNKNOWN_LABEL, "Localized only")
    .replace(STAGE_ORDER)
    .cast(pl.Int8)
    .alias(STAGE_COL)
)

# ============================================================
# Consolidation -> One-Hot Encoding for HISTOLOGY variable
# ============================================================
HISTOLOGY_MAP = {
    "8140-8389: adenomas and adenocarcinomas": "Adenocarcinomas",
    "8500-8549: ductal and lobular neoplasms": "Ductal and lobular",
    "8010-8049: epithelial neoplasms, NOS": "Epithelial NOS",
    "8050-8089: squamous cell neoplasms": "Squamous cell",
    "8120-8139: transitional cell papillomas and carcinomas": "Transitional cell",
    "8000-8009: unspecified neoplasms": "Unspecified neoplasms",
    "8720-8799: nevi and melanomas": "Melanomas",
    "8440-8499: cystic, mucinous and serous neoplasms": "Cystic/mucinous/serous",
    "8560-8579: complex epithelial neoplasms": "Complex epithelial",
    "8930-8999: complex mixed and stromal neoplasms": "Complex mixed and stromal",
    "8550-8559: acinar cell neoplasms": "Acinar cell",
    "8890-8929: myomatous neoplasms": "Myomatous",
}

HIST_COL = "histology"

seer_encoded = seer_encoded.with_columns(
    pl.col(HIST_COL).replace_strict(HISTOLOGY_MAP, default="Other").alias(HIST_COL)
)

seer_encoded = seer_encoded.to_dummies(columns=[HIST_COL])

ref_col = f"{HIST_COL}_Adenocarcinomas"
seer_encoded = seer_encoded.drop(ref_col)

seer_encoded = seer_encoded.with_columns(
    [pl.col(c).cast(pl.Int8) for c in seer_encoded.columns if c.startswith(HIST_COL)]
)

# ==============================================================================
# One-Hot Encoding for MARITAL_STATUS and flagging uknown with MARITAL_UNKNOWN
# ==============================================================================
MARITAL_COL = "marital_status"
UNKNOWN_MARITAL = "Unknown"

seer_encoded = seer_encoded.with_columns(
    (pl.col(MARITAL_COL) == UNKNOWN_MARITAL).cast(pl.Int8).alias("marital_unknown")
)

seer_encoded = seer_encoded.with_columns(
    pl.col(MARITAL_COL)
    .replace(
        {
            UNKNOWN_MARITAL: "Married (including common law)",  # mode imputation
        }
    )
    .alias(MARITAL_COL)
)

seer_encoded = seer_encoded.to_dummies(columns=[MARITAL_COL])

ref_col = f"{MARITAL_COL}_Married (including common law)"
seer_encoded = seer_encoded.drop(ref_col)

seer_encoded = seer_encoded.with_columns(
    [pl.col(c).cast(pl.Int8) for c in seer_encoded.columns if c.startswith(MARITAL_COL)]
)

# ======================================================================================================
# Imputation and normalization for TUMOR_SIZE variable, and flagging for UNKNOWN MASS and TUMOR NO MASS
# ======================================================================================================
TUMOR_COL = "tumor_size"
UNKNOWN_SENTINELS = [
    "Unknown or size unreasonable (includes any tumor sizes 401-989)",
    "Tumor Size Not Consistent Over Time or Not Applicable for this Site",
    "998 (site-specific code)",
]
MICROSCOPIC = "990 (microscopic focus)"
NO_PRIMARY = "000 (no evidence of primary tumor)"

seer_encoded = seer_encoded.with_columns(
    [
        pl.col(TUMOR_COL).is_in(UNKNOWN_SENTINELS).cast(pl.Int8).alias("tumor_size_unknown"),
        (pl.col(TUMOR_COL) == NO_PRIMARY).cast(pl.Int8).alias("tumor_size_no_mass"),
    ]
)

seer_encoded = seer_encoded.with_columns(
    pl.when(pl.col(TUMOR_COL).is_in(UNKNOWN_SENTINELS))
    .then(None)
    .when(pl.col(TUMOR_COL) == MICROSCOPIC)
    .then(pl.lit(0.5))
    .when(pl.col(TUMOR_COL) == NO_PRIMARY)
    .then(pl.lit(0.0))
    .otherwise(pl.col(TUMOR_COL))
    .cast(pl.Float32)
    .alias(TUMOR_COL)
)

seer_encoded = seer_encoded.with_columns(
    pl.col(TUMOR_COL).fill_null(strategy="mean").alias(TUMOR_COL)
)

t_min = seer_encoded[TUMOR_COL].min()
t_max = seer_encoded[TUMOR_COL].max()
seer_encoded = seer_encoded.with_columns(
    ((pl.col(TUMOR_COL) - t_min) / (t_max - t_min)).cast(pl.Float32).alias(TUMOR_COL)
)

seer_encoded

In [ ]:
# ======================================================================== #
#                           OUTCOME EVENT CODING                           #
# ======================================================================== #
seer_encoded = seer_encoded.with_columns(
    pl.when(
        (pl.col("cause_specific_death") == "Dead (attributable to this cancer dx)")
        & (pl.col("vital_status") == "Dead")
    )
    .then(0)
    .when(
        (pl.col("cause_specific_death") != "Dead (attributable to this cancer dx)")
        & (pl.col("vital_status") == "Dead")
    )
    .then(1)
    .otherwise(2)
    .alias("outcome")
).drop(["vital_status", "cause_specific_death", "other_cause_death"])

# Dropping rows introducing time ambiguity
seer_encoded = seer_encoded.filter(
    pl.col("survival_months_flag")
    != "Incomplete dates are available and there could be zero days of follow-up"
).drop("survival_months_flag")


seer_encoded = seer_encoded.rename({"survival_months": "time"})

seer_encoded

In [ ]:
cancer_col_name_map = {
    "Breast": "Breast",
    "Prostate": "Corpus",
    "Lung And Bronchus": "Kidney Parenchyma",
    "Colon And Rectum (Excluding Appendix)": "Melanoma",
    "Melanoma Of The Skin": "Lung & Bronchus",
    "Urinary Bladder": "Pancreas",
    "Kidney Parenchyma": "Prostate",
    "Corpus": "Thyroid",
    "Pancreas": "Urinary Bladder",
    "Thyroid": "Colorectal",
}

In [ ]:
def get_cancer_cohort(df, cancer_type, sample_n):
    cohort_df = df.filter(pl.col("site_recode") == cancer_type)
    sampled_cohort_df, _ = train_test_split(
        cohort_df, train_size=sample_n, stratify=cohort_df["outcome"], random_state=SEED
    )
    cols = sampled_cohort_df.columns
    X = sampled_cohort_df.select(
        [col for col in cols if col not in ["time", "outcome", "site_recode"]]
    )
    y = sampled_cohort_df.select(["outcome", "time"])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y["outcome"], random_state=SEED
    )
    return X_train, X_test, y_train, y_test


SAMPLE_N = 10000
cancer_event_rate_dict = {
    "cancer_type": [],
    "n_primary_deaths": [],
    "n_other_deaths": [],
    "n_survived": [],
}

for cancer_type in top10cancers:
    X_train, X_test, y_train, y_test = get_cancer_cohort(seer_encoded, cancer_type, SAMPLE_N)

    cancer_event_rate_dict["cancer_type"].append(cancer_col_name_map[cancer_type])
    cancer_event_rate_dict["n_primary_deaths"].append(
        y_train.filter(pl.col("outcome") == 0).height + y_test.filter(pl.col("outcome") == 0).height
    )
    cancer_event_rate_dict["n_other_deaths"].append(
        y_train.filter(pl.col("outcome") == 1).height + y_test.filter(pl.col("outcome") == 1).height
    )
    cancer_event_rate_dict["n_survived"].append(
        y_train.filter(pl.col("outcome") == 2).height + y_test.filter(pl.col("outcome") == 2).height
    )

    cancer_name = cancer_type.lower().replace(" ", "_")
    cancer_dir_path = CANCER_SPECIFIC_DATA_PATH / cancer_name

    for split in ["train", "test"]:
        if not os.path.exists(cancer_dir_path / split):
            os.makedirs(cancer_dir_path / split)

    X_train.write_csv(cancer_dir_path / "train" / f"X_{cancer_name}.csv")
    X_test.write_csv(cancer_dir_path / "test" / f"X_{cancer_name}.csv")
    y_train.write_csv(cancer_dir_path / "train" / f"y_{cancer_name}.csv")
    y_test.write_csv(cancer_dir_path / "test" / f"y_{cancer_name}.csv")

In [ ]:
cancer_event_rate_dict

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

C_DEATH = "#1a1a6e"
C_OTHER = "#6b8cda"
C_ALIVE = "#b8cef5"

fig, ax = plt.subplots(figsize=(16, 6))

x = np.arange(len(cancer_event_rate_dict["cancer_type"]))
width = 0.55

bars_death = ax.bar(
    x,
    cancer_event_rate_dict["n_primary_deaths"],
    width,
    color=C_DEATH,
    label="Cancer-specific death",
)
bars_other = ax.bar(
    x,
    cancer_event_rate_dict["n_other_deaths"],
    width,
    color=C_OTHER,
    label="Other-cause death",
    bottom=cancer_event_rate_dict["n_primary_deaths"],
)
bars_alive = ax.bar(
    x,
    cancer_event_rate_dict["n_survived"],
    width,
    color=C_ALIVE,
    label="Censored / Alive",
    bottom=[
        d + o
        for d, o in zip(
            cancer_event_rate_dict["n_primary_deaths"], cancer_event_rate_dict["n_other_deaths"]
        )
    ],
)

MIN_LABEL = 200


def add_labels(bars, bottoms, values, text_color):
    for bar, bottom, val in zip(bars, bottoms, values):
        if val >= MIN_LABEL:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bottom + val / 2,
                f"{val:,}",
                ha="center",
                va="center",
                fontsize=8.5,
                fontweight="bold",
                color=text_color,
            )


zeros = [0] * len(x)
add_labels(bars_death, zeros, cancer_event_rate_dict["n_primary_deaths"], "white")
add_labels(
    bars_other,
    cancer_event_rate_dict["n_primary_deaths"],
    cancer_event_rate_dict["n_other_deaths"],
    "white",
)
add_labels(
    bars_alive,
    [
        d + o
        for d, o in zip(
            cancer_event_rate_dict["n_primary_deaths"], cancer_event_rate_dict["n_other_deaths"]
        )
    ],
    cancer_event_rate_dict["n_survived"],
    C_DEATH,
)

ax.set_xticks(x)
ax.set_xticklabels(cancer_event_rate_dict["cancer_type"], fontsize=9.5)
ax.set_ylim(0, SAMPLE_N * 1.05)
ax.set_yticks(range(0, SAMPLE_N + 1, 2000))
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f"{int(v/1000)}k" if v >= 1000 else str(int(v)))
)
ax.set_ylabel("Number of Patients", fontsize=11)
ax.set_title("Outcome Distribution – Top 10 Cancer Types (SEER, 2004–2021)", fontsize=15)

ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, color="lightgrey", linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

legend_patches = [
    mpatches.Patch(color=C_DEATH, label="Cancer-specific death"),
    mpatches.Patch(color=C_OTHER, label="Other-cause death"),
    mpatches.Patch(color=C_ALIVE, label="Censored / Alive"),
]
ax.legend(handles=legend_patches, frameon=True, fontsize=9.5, loc="upper left")

plt.tight_layout()
plt.savefig(GRAPH_PATH / "cohort_outcome_distribution.png", dpi=1000)
plt.show()